In [1]:
################# Initial Conditions File with Sea Ice! ###################
# The purpose of this script is to create an initial conditions 
# file for the Alaskan Beaufort Sea shelf that include sea ice variables. 
# Some of the values are taken from 
# the open boundary and initial conditions files while others
# are set to small values (maybe).
#
# Notes:
# - This script does not have sediment but adds sea ice. It is also for
#   the summer of 2020 but can easily be changed depending what inputs
#   are given.
# - We want to start the sea ice model at September 1, 2019, so this will 
#   use the time step that matches that for initial conditions from all 
#   inputs and the time in seconds will be updated 
# - This script is set up to do both the 20- and 30-vertical layer versions
#   for initial conditions by just commenting things in/out
############################################################################

In [2]:
# Load in the packages
from netCDF4 import Dataset
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
from cftime import num2date, date2num
import xesmf as xe
import math
#import ESMF
#import time

In [3]:
# TEST
ar = np.arange(0,11,1)
print(ar)
print(len(ar))
print(ar[0:10])
print(ar[10])

[ 0  1  2  3  4  5  6  7  8  9 10]
11
[0 1 2 3 4 5 6 7 8 9]
10


In [4]:
# Open grid file to read in dimensions
nc_f1 = '/global/homes/b/bundzis/Projects/Beaufort_ROMS_2020_test_nosed/Include/KakAKgrd_shelf_big010_smooth006_thin_sponge.nc' # UPDATE PATH
nc1 = Dataset(nc_f1, 'r')

In [5]:
# Open grid again using xarray since it's easier
grid = xr.open_dataset('/global/homes/b/bundzis/Projects/Beaufort_ROMS_2020_test_nosed/Include/KakAKgrd_shelf_big010_smooth006_thin_sponge.nc')

In [6]:
# Open the sed_bed_toy test case restrat_mix_ini.nc file to pull values from there
#nc2 = xr.open_dataset('/Users/brun1463/Desktop/Research_Lab/Kaktovik_Alaska/Code/restrat_mix_ini.nc')
#nc_f2 = '/projects/brun1463/ROMS/Kakak3_Alpine_2020/Scripts/Initial_conds/restrat_mix_ini.nc' # UPDATE PATH
#nc2 = Dataset(nc_f2, 'r')

In [7]:
# Read in dimensions from the grid
xi_psi_tmp = nc1.variables['xi_psi']
xi_rho_tmp = nc1.variables['xi_rho']
xi_u_tmp = nc1.variables['xi_u']
xi_v_tmp = nc1.variables['xi_v']
eta_psi_tmp = nc1.variables['eta_psi']
eta_rho_tmp = nc1.variables['eta_rho']
eta_u_tmp = nc1.variables['eta_u']
eta_v_tmp = nc1.variables['eta_v']
bath_tmp = nc1.variables['h']

# Save dimensions
Lp = len(xi_rho_tmp)
Mp = len(eta_rho_tmp)
Lm = Lp-2
Mm = Mp-2
L = Lm+1
M = Mm+1

In [8]:
nc1

<class 'netCDF4._netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    gridname: cg_Kak
    type: ROMS grid file
    history: 
    Conventions: CF-1.2
    Institution: University of Colorado Boulder
    date: 2022-07-11
    grid_mapping_name: polar_stereographic
    latitude_of_projection_origin: 90.0
    straight_vertical_longitude_from_pole: 73.93517116325103
    standard_parallel: 60.0
    false_easting: 0
    false_northing: 0
    dx: 500
    ellipsoid: sphere
    earth_radius: 6371000.0
    dimensions(sizes): one(1), xi_rho(608), eta_rho(206), xi_u(607), eta_u(206), xi_v(608), eta_v(205), xi_psi(607), eta_psi(205), string1(1)
    variables(dimensions): float64 xl(one), float64 el(one), float64 depthmax(one), float64 depthmin(one), float64 xi_rho(xi_rho), float64 eta_rho(eta_rho), float64 xi_u(xi_u), float64 eta_u(eta_u), float64 xi_v(xi_v), float64 eta_v(eta_v), float64 xi_psi(xi_psi), float64 eta_psi(eta_psi), float64 x_rho(eta_rho, xi_rho), float64 y_rho(eta_r

In [9]:
# Pull out the angle to rotate the currents to match the grid's u,v
phi = grid.angle[0,0].values # radians 

In [10]:
# Fill the missing dimensions with data from sed_bed_toy test case
#s_w_tmp = nc2.dimensions['s_w']
#s_w_tmp = np.arange(0, 21, 1) # 20-vertical OG 
s_w_tmp = np.arange(0, 31, 1) # 30-vertical NEW
s_w_tmp_len = len(s_w_tmp)

#tracer_tmp = nc2.dimensions['tracer']
#tracer_tmp_len = len(tracer_tmp)
tracer_tmp_len = 2 # (5 sediment, salt, temp) # not sure is passive tracers need to be included here...

#s_rho_tmp = 20 # 20-vertical OG
s_rho_tmp = 30 # 30-vertical NEW

Nbed_tmp = 0 # 10, 41

one_tmp = 1

two_tmp = 2

# temporary time length
#time_tmp = 1
time_tmp = 620614800  # 620614800 = September 1, 2019, hour 1; 646880400 = July 1, 2020 hour 1;  # 615258000 # now set to hour 1; hour 0: 615254400.0 (July 1, hour 1, 2020)
time_tmp_len = 1

# set ocean_time (time since initializtion)
ocean_time_tmp = time_tmp
#print('ocean_time: ', ocean_time_tmp[0])

In [11]:
# Assign values for variables
# These will need to change when we change the vertical grid 
# theta_b (S-coordinate bottom control parameter)
#theta_b_tmp = np.asarray(3) # 3.0, 20-vertical OG
theta_b_tmp = np.asarray(3) # 3.0, 30-vertical NEW

# theta_s (S-coordinate surface control parameter)
#theta_s_tmp = np.asarray(1) # 1.0, 20-vertical OG
theta_s_tmp = np.asarray(5) # 5.0, 30-vertical NEW

# Tcline (S-coordinate surface/bottom layer width)
Tcline_tmp = np.asarray(5.0)

# hc (S-coordinate parameter, critical depth)
hc_tmp = np.asarray(5.0)

In [12]:
# ----------------- Use OBC files to find initial values ----------------
# For variables that we have info on from open boundary condition (OBC)
# files, read in those values into this to set as initial conditions

#### Salinity

In [13]:
# Salt (salinity)
# Read in the OBC file 
#salt_clm_obc = xr.open_dataset('/Users/brun1463/Desktop/Research_Lab/Kaktovik_Alaska/ROMS_interpolate_data_scripts_output/final_boundary_conditions/attempt007/salt_clm_009.nc') # UPDATE PATH
#salt_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_test_sed_scratch/Model_Inputs/Bry_Clm_Conds/Attempt001/salt_clm_001.nc') # UPDATE PATH
#salt_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/salt_clm_2019_2024_20vert_001.nc')
salt_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/salt_clm_2019_2024_30vert_001.nc')


In [14]:
# Find the time that matches September 1, 2019, hour 1
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in salt_clm_obc.salt_time]

# Try different indices
salt_time_idx = 243
print('salt datetime: ', datetimes[salt_time_idx])
print('salt_clm_obc time: ', salt_clm_obc.salt_time[salt_time_idx].values)

salt datetime:  2019-09-01 01:00:00
salt_clm_obc time:  620614800.0


In [15]:
# Make an array to hold these initial values
salt_tmp = np.empty((time_tmp_len, s_rho_tmp, Mp, Lp))

# Fill the array
salt_tmp[:,:,:,:] = salt_clm_obc.salt[salt_time_idx,:,:,:].values

# Check that this worked
print('salt: ', salt_tmp[0,0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(salt_clm_obc)

salt:  0.0


#### Temperature

In [16]:
# Temperature (potential temperature)
# Read in the OBC file 
#temp_clm_obc = xr.open_dataset('/Users/brun1463/Desktop/Research_Lab/Kaktovik_Alaska/ROMS_interpolate_data_scripts_output/final_boundary_conditions/attempt007/temp_clm_008.nc') # UPDATE PATH
#temp_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_test_sed_scratch/Model_Inputs/Bry_Clm_Conds/Attempt001/temp_clm_001.nc') # UPDATE PATH
#temp_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/temp_clm_2019_2024_20vert_001.nc')
temp_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/temp_clm_2019_2024_30vert_001.nc')


In [17]:
# Find the time that matches September 1, 2019, hour 1
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in temp_clm_obc.temp_time]

# Try different indices
temp_time_idx = 243
print('temp datetime: ', datetimes[temp_time_idx])
print('temp_clm_obc time: ', temp_clm_obc.temp_time[temp_time_idx].values)

temp datetime:  2019-09-01 01:00:00
temp_clm_obc time:  620614800.0


In [18]:
# Make an array to hold these initial values
temp_tmp = np.empty((time_tmp_len, s_rho_tmp, Mp, Lp))

# Fill the array
temp_tmp[:,:,:,:] = temp_clm_obc.temp[temp_time_idx,:,:,:].values

# Check that this worked
print('temp: ', temp_tmp[0,0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(temp_clm_obc)

temp:  0.0


#### Zeta

In [19]:
# Zeta (free surface)
# Read in the OBC file 
#zeta_clm_obc = xr.open_dataset('/Users/brun1463/Desktop/Research_Lab/Kaktovik_Alaska/ROMS_interpolate_data_scripts_output/final_boundary_conditions/attempt007/zeta_clm_005.nc') # UPDATE PATH
#zeta_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_test_sed_scratch/Model_Inputs/Bry_Clm_Conds/Attempt001/zeta_clm_2020_004.nc') # UPDATE PATH
zeta_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/zeta_clm_2019_2024_001.nc')

In [20]:
# Find the time that matches September 1, 2019, hour 1
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in zeta_clm_obc.zeta_time]

# Try different indices
zeta_time_idx = 243
print('zeta datetime: ', datetimes[zeta_time_idx])
print('zeta_clm_obc time: ', zeta_clm_obc.zeta_time[zeta_time_idx].values)

zeta datetime:  2019-09-01 01:00:00
zeta_clm_obc time:  620614800.0


In [21]:
# Make an array to hold these initial values
zeta_tmp = np.empty((time_tmp_len, Mp, Lp))

# Fill the array
zeta_tmp[:,:,:] = zeta_clm_obc.zeta[zeta_time_idx,:,:].values

# Check that this worked
print('zeta: ', zeta_tmp[0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(zeta_clm_obc)

zeta:  -0.0


#### U

In [22]:
# u (u momentum)
# Read in the OBC file 
#u_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/u_currents_clm_2019_2024_20vert_001.nc')
#u_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/u_currents_clm_2019_2024_20vert_001_short_nonan.nc')
#u_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/u_currents_clm_2019_2024_30vert_001.nc')
u_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/u_currents_clm_2019_2024_30vert_001_nonan.nc')

In [23]:
# Find the time that matches September 1, 2019, hour 1
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in u_clm_obc.v3d_time]

# Try different indices
u_time_idx = 243 # 243, 0
print('u datetime: ', datetimes[u_time_idx])
print('u_clm_obc time: ', u_clm_obc.v3d_time[u_time_idx].values)

u datetime:  2019-09-01 01:00:00
u_clm_obc time:  620614800.0


In [24]:
# Make an array to hold these initial values
u_tmp = np.empty((time_tmp_len, s_rho_tmp, Mp, L))

# Fill the array
u_tmp[:,:,:,:] = u_clm_obc.u[u_time_idx,:,:,:].values

# Check that this worked
print('u: ', u_tmp[0,0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(u_clm_obc)

u:  0.0


#### Ubar

In [25]:
# ubar (vertically integrated u-momentum component)
# Read in the OBC file 
#ubar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/ubar_currents_clm_2019_2024_20vert_001.nc')
#ubar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/ubar_currents_clm_2019_2024_20vert_001_short_nonan.nc')
#ubar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/ubar_currents_clm_2019_2024_30vert_001.nc')
ubar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/ubar_currents_clm_2019_2024_30vert_001_nonan.nc')


In [26]:
# Find the time that matches September 1, 2019, hour 1
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in ubar_clm_obc.v2d_time]

# Try different indices
ubar_time_idx = 243 # 243, 0
print('ubar datetime: ', datetimes[ubar_time_idx])
print('ubar_clm_obc time: ', ubar_clm_obc.v2d_time[ubar_time_idx].values)

ubar datetime:  2019-09-01 01:00:00
ubar_clm_obc time:  620614800.0


In [27]:
# Make an array to hold these initial values
ubar_tmp = np.empty((time_tmp_len, Mp, L))

# Fill the array
ubar_tmp[:,:,:] = ubar_clm_obc.ubar[ubar_time_idx,:,:].values

# Check that this worked
print('ubar: ', ubar_tmp[0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(ubar_clm_obc)

ubar:  0.0


#### V

In [28]:
# v (v momentum)
# Read in the OBC file 
#v_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/v_currents_clm_2019_2024_20vert_001.nc')
#v_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/v_currents_clm_2019_2024_20vert_001_short_nonan.nc')
#v_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/v_currents_clm_2019_2024_30vert_001.nc')
v_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/v_currents_clm_2019_2024_30vert_001_nonan.nc')


In [29]:
v_clm_obc

<xarray.Dataset> Size: 124GB
Dimensions:   (v3d_time: 2075, s_rho: 30, eta_rho: 206, xi_rho: 608,
               eta_v: 205, xi_v: 608)
Coordinates:
  * xi_rho    (xi_rho) float64 5kB 0.0 1.0 2.0 3.0 ... 604.0 605.0 606.0 607.0
  * eta_rho   (eta_rho) float64 2kB 0.0 1.0 2.0 3.0 ... 202.0 203.0 204.0 205.0
  * xi_v      (xi_v) float64 5kB 0.0 1.0 2.0 3.0 4.0 ... 604.0 605.0 606.0 607.0
  * eta_v     (eta_v) float64 2kB 0.0 1.0 2.0 3.0 ... 201.0 202.0 203.0 204.0
  * s_rho     (s_rho) float64 240B 0.0 1.0 2.0 3.0 4.0 ... 26.0 27.0 28.0 29.0
  * v3d_time  (v3d_time) float64 17kB 5.996e+08 5.997e+08 ... 7.788e+08
Data variables:
    z_rho     (v3d_time, s_rho, eta_rho, xi_rho) float64 62GB ...
    v         (v3d_time, s_rho, eta_v, xi_v) float64 62GB ...
Attributes:
    gridname:     KakAKgrd_shelf_big010_smooth006_thin_sponge.nc
    type:         ROMS grid vertically interpolated HYCOM v currents climatology
    history:      Created by Brianna Undzis
    Conventions:  CF
    Institution:  University of Colorado Boulder
    date:         2025-08-27 00:13:29.712089

In [30]:
# Find the time that matches September 1, 2019, hour 1
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in v_clm_obc.v3d_time]

# Try different indices
v_time_idx = 243 # 243, 0
print('v datetime: ', datetimes[v_time_idx])
print('v_clm_obc time: ', v_clm_obc.v3d_time[v_time_idx].values)

v datetime:  2019-09-01 01:00:00
v_clm_obc time:  620614800.0


In [31]:
# Make an array to hold these initial values
v_tmp = np.empty((time_tmp_len, s_rho_tmp, M, Lp))

# Fill the array
v_tmp[:,:,:,:] = v_clm_obc.v[v_time_idx,:,:,:].values

# Check that this worked
print('v: ', v_tmp[0,0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(v_clm_obc)

v:  0.0


#### Vbar

In [32]:
# vbar (vertically integrated v-momentum component)
# Read in the OBC file 
#vbar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/vbar_currents_clm_2019_2024_20vert_001.nc')
#vbar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/vbar_currents_clm_2019_2024_20vert_001_short_nonan.nc')
#vbar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/vbar_currents_clm_2019_2024_30vert_001.nc')
vbar_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/vbar_currents_clm_2019_2024_30vert_001_nonan.nc')


In [33]:
# Find the time that matches September 1, 2019, hour 1
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in vbar_clm_obc.v2d_time]

# Try different indices
vbar_time_idx = 243 # 243, 0
print('vbar datetime: ', datetimes[vbar_time_idx])
print('vbar_clm_obc time: ', vbar_clm_obc.v2d_time[vbar_time_idx].values)

vbar datetime:  2019-09-01 01:00:00
vbar_clm_obc time:  620614800.0


In [34]:
# Make an array to hold these initial values
vbar_tmp = np.empty((time_tmp_len, M, Lp))

# Fill the array
vbar_tmp[:,:,:] = vbar_clm_obc.vbar[vbar_time_idx,:,:].values

# Check that this worked
print('vbar: ', vbar_tmp[0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(vbar_clm_obc)

vbar:  0.0


#### Sea Ice

In [35]:
# Sea ice
# Read in the OBC file 
sea_ice_clm_obc = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/Forcing_files/Bryclm/Attempt001/ice_clm_2019_2024_001.nc')

In [36]:
# Find the time that matches September 1, 2019, hour 1 # PICK UP HERE
# Convert it to datetime first 
# Define the reference date
reference_date = datetime(2000, 1, 1)

# Convert each to datetime
datetimes = [reference_date + timedelta(seconds=int(s)) for s in sea_ice_clm_obc.ocean_time]

# Try different indices
sea_ice_time_idx = 243
print('sea_ice datetime: ', datetimes[sea_ice_time_idx])
print('sea_ice_clm_obc time: ', sea_ice_clm_obc.ocean_time[sea_ice_time_idx].values)

sea_ice datetime:  2019-09-01 01:00:00
sea_ice_clm_obc time:  620614800.0


In [37]:
# Make empty arrays to hold the initial values
# Ice area fraction (Aice)
# fraction of cell covered by ice
Aice_tmp = np.empty((time_tmp_len, Mp, Lp))

# Ice thickness (ice_thickness)
# average ice thickness in cell
ice_thickness_tmp = np.empty((time_tmp_len, Mp, Lp))

# Ice u-velocity (Uice)
# u-component of ice velocity 
Uice_tmp = np.empty((time_tmp_len, Mp, L))

# Ice v-velocity (Vice)
# v-component of ice velocity
Vice_tmp = np.empty((time_tmp_len, M, Lp))


# Fill these with the data from the clm file
Aice_tmp[:,:,:] = sea_ice_clm_obc.Aice[sea_ice_time_idx,:,:].values
ice_thickness_tmp[:,:,:] = sea_ice_clm_obc.ice_thickness[sea_ice_time_idx,:,:].values
Uice_tmp[:,:,:] = sea_ice_clm_obc.Uice[sea_ice_time_idx,:,:].values
Vice_tmp[:,:,:] = sea_ice_clm_obc.Vice[sea_ice_time_idx,:,:].values

# Check that this worked
print('Aice: ', Aice_tmp[0,0,0])

# Remove the salt_clm_OBC file for memory sake
del(sea_ice_clm_obc)

Aice:  0.0


In [38]:
# ----------------- Use sustr/svstr files to find initial values ----------------
# For surface stress variables that we have info on from HYCOM
# files, read in those values into this to set as initial conditions
# This is not needed so ignore this for now

In [39]:
# # sustr (surface u-momentum stress)
# # Read in the sustr forcing file 
# sustr_frc = xr.open_dataset('/Users/brun1463/Desktop/Research_Lab/Kaktovik_Alaska/Code/sustr_forcing_file_kaktovik_shelf_hycom_data_0002.nc')

# # Make an array to hold these initial values
# sustr_tmp = np.empty((time_tmp_len, MP, L))

# # Fill the array
# sustr_tmp[:,:,:] = sustr_frc.sustr[0,:,:].values

# # Check that this worked
# print('sustr: ', sustr_tmp[0,0,0])

# # Remove the file for memory sake
# del(sustr_frc)

In [40]:
# # svstr (surface v-momentum stress)
# # Read in the svstr forcing file 
# svstr_frc = xr.open_dataset('/Users/brun1463/Desktop/Research_Lab/Kaktovik_Alaska/Code/svstr_forcing_file_kaktovik_shelf_hycom_data_0002.nc')

# # Make an array to hold these initial values
# svstr_tmp = np.empty((time_tmp_len, M, Lp))

# # Fill the array
# svstr_tmp[:,:,:] = svstr_frc.svstr[0,:,:].values

# # Check that this worked
# print('svstr: ', svstr_tmp[0,0,0])

# # Remove the file for memory sake
# del(svstr_frc)

In [41]:
# --------------------- End use of OBC to fill values ----------------------

In [42]:
# Variables taken from test case or other ROMS output
# Load in ROMS output
#roms_out = xr.open_dataset('/Users/brun1463/Desktop/Research_Lab/Kaktovik_Alaska/model_output/Full_run_0003_sponge_swell/ocean_his_biggrid010_gridwindsiniwaves_rivs_si_smooth006_nobulk_chaflaradnudclm_dbsed0007_0007.nc')
roms_out = xr.open_dataset('/pscratch/sd/b/bundzis/Beaufort_ROMS_2020_dvd_myroms_ice_scratch/roms_his_beaufort_2020_dvd_myroms_blkflx_32x8_002_0001.nc')


# Cs_r (S-coordinate stretching curves at RHO-points)
# Use value from ROMS output
Cs_r_tmp = roms_out.Cs_r.values
print('Cs_r: ', Cs_r_tmp[0])

# Cs_w (S-coordinate stretching curves at W-points)
# Use value from ROMS output
Cs_w_tmp = roms_out.Cs_w.values
print('Cs_w: ', Cs_w_tmp[0])

# Not sure what to set these to so let's try not including them and 
# see what happens...
# # sc_r (S-coordinate at RHO-points)
# # NOT SURE WHAT TO DO HERE
# sc_r_tmp2 = nc2.variables['sc_r']
# sc_r_tmp = np.full((s_rho_tmp), sc_r_tmp2[0], dtype='f8')
# print('sc_r: ', sc_r_tmp[0])

# # sc_w (S-coordinate at W-points)
# # NOT SURE WHAT TO DO HERE
# sc_w_tmp2 = nc2.variables['sc_w']
# sc_w_tmp = np.full((s_w_tmp_len), sc_w_tmp2[0], dtype='f8')
# print('Cs_r: ', Cs_r_tmp[0])

# Delete the roms output to save space
del(roms_out)

Cs_r:  -0.9982328699255587
Cs_w:  -1.0


In [43]:
Cs_r_tmp

array([-0.99823287, -0.99293027, -0.98420035, -0.97033365, -0.9492267 ,
       -0.91860081, -0.87637397, -0.82116529, -0.75282454, -0.67279613,
       -0.58412052, -0.49100013, -0.39806374, -0.30961308, -0.22911375,
       -0.1590283 , -0.1009154 , -0.05563959, -0.02355971, -0.0046374 ])

### Sea Ice!
Make some sea ice variables to put in the initial conditions file!  
This section only needs to be used if the ice things are not being read in from a climatology file.

In [44]:
# Sea Ice

# Load in the ice clm file to read the variables from there? (above)

# Or load in the OG data directly and pull from there
# Remember the time difference...subtract 8/9 hours...
# hycom_ice_data = xr.open_dataset('/pscratch/sd/b/bundzis/External_data/HYCOM_data/hycom_daily_ncss/hycom_ice_2019_2024.nc')
# hycom_ice_data

In [45]:
# # HYCOM number of lats
# hycom_lat_len = len(hycom_ice_data.lat.values)
# #print('hycom_lat_len', hycom_lat_len)

# # HYCOM number of lons
# hycom_lon_len = len(hycom_ice_data.lon.values)
# #print('hycom_lon_len', hycom_lon_len)

# # HYCOM time len (not really releveant since we just need one time but yolo)
# hycom_time_len = len(hycom_ice_data.time.values)

In [46]:
# # Find the time that is the first time step for our concerns 
# # July 1, 2020 hour 1 in AKDT so July 1, hour 9 in UTC
# print(hycom_ice_data.time[0].values)
# print(hycom_ice_data.time[4307].values)
# print(hycom_ice_data.time[-1].values)

# # So for starting in July 1, 2020, hour 1 (AKDT), want time index 4307

In [47]:
# # ** Only needed if not reading from clm file **
# # Grid's lat/lon is in different convention than HYCOM lat/lon 
# # need to make HYCOM match grid's lat lon convention 
# hycom_ice_data['lon_180'] = -(360 - hycom_ice_data.lon.values)

In [48]:
# # ** Only needed if not reading from clm file **
# # Now before we can regird and such, we need to rotate the HYCOM data
# # Define a function to do this rotation 
# def rotate2(point, angle):
#     """
#     Rotate a point counterclockwise by a given angle around a given origin.

#     The angle should be given in radians.
    
#     This equation can be verified here: https://www.myroms.org/forum/viewtopic.php?f=3&t=295 
#     where we are rotating from lon,lat to the same lon,lat (rotating HYCOM data about an angle)
#     """
#     px, py = point

#     qx = math.cos(angle) * (px) - math.sin(angle) * (py)
#     qy = math.sin(angle) * (px) + math.cos(angle) * (py)
#     return qx, qy

In [49]:
# # ** Only needed if not reading from clm file **
# # Apply this rotation to all u and v in HYCOM
# # Declare empty arrays to fill with the rotated values 
# ice_u_rot = np.empty((hycom_time_len,hycom_lat_len,hycom_lon_len)) # only one time since this is initial conditions file 
# ice_v_rot = np.empty((hycom_time_len,hycom_lat_len,hycom_lon_len))

# # Rotate the currents and save the values to the arrays; this takes ~1 minute
# ice_u_rot[:,:,:], ice_v_rot[:,:,:] = rotate2((hycom_ice_data.siu[:,:,:].values, hycom_ice_data.siv[:,:,:].values), phi)

# # Check to see if this rotation worked 
# # # check u
# # print('Unrotated u: ', hycom_ice_data.ssu[0,80,200].values)
# # print('Expected rotated u: ', math.cos(phi) * (hycom_ice_data.ssu[0,80,200].values) - math.sin(phi) * (hycom_ice_data.ssv[0,80,200].values))
# # print('Actual Rotated u: ', ice_u_rot[0,80,200])

# # # check v
# # print('Unrotated v: ', hycom_ice_data.ssv[0,80,200].values)
# # print('Expected rotated v: ', math.sin(phi) * (hycom_ice_data.ssu[0,80,200].values) + math.cos(phi) * (hycom_ice_data.ssv[0,80,200].values))
# # print('Actual Rotated v: ', ice_v_rot[0,80,200])

In [50]:
# # This rotation seemed to work correctly and was super fast!! So yay! 
# # Now make these rotated currents part of the HYCOM data set
# # ex: ds['nmap'] = (('y', 'x'), nmap)
# hycom_ice_data['ice_u_rot'] = (('time', 'lat', 'lon'), ice_u_rot)
# hycom_ice_data['ice_v_rot'] = (('time', 'lat', 'lon'), ice_v_rot)

In [51]:
# # Set the input and output grids, and sepcify the lat/lon
# # Since we are looking at u for now, we will use lon_u and lat_u as the primary lat/lon for the grid 
# # Input grid (HYCOM)
# ds_in_hycom = hycom_ice_data.copy() # need to use lon_180 for this grid 
# ds_in_hycom['lon_360'] = ds_in_hycom.lon.values
# ds_in_hycom['lon'] = ds_in_hycom.lon_180.values

# # Output grid u (ROMS u grid, but keeps HYCOM vertical levels)
# ds_out_u = grid.copy()
# ds_out_u['lat'] = (('eta_u', 'xi_u'), ds_out_u.lat_u.values)
# ds_out_u['lon'] = (('eta_u', 'xi_u'), ds_out_u.lon_u.values)

# # Output grid v (ROMS, but keeps HYCOM vertical levels)
# ds_out_v = grid.copy()
# ds_out_v['lat'] = (('eta_v', 'xi_v'), ds_out_v.lat_v.values)
# ds_out_v['lon'] = (('eta_v', 'xi_v'), ds_out_v.lon_v.values)

# # Output grid (ROMS rho grid)
# #ds_out_rho = grid_vertical
# ds_out_rho = grid.copy()
# ds_out_rho['lat'] = (('eta_rho', 'xi_rho'), ds_out_rho.lat_rho.values)
# ds_out_rho['lon'] = (('eta_rho', 'xi_rho'), ds_out_rho.lon_rho.values)

# # Add masks 
# # ex: ds["mask"] = xr.where(~np.isnan(ds["zeta"].isel(ocean_time=0)), 1, 0)
# # Input grid (HYCOM)
# # this is only a surface mask - which is what we want
# ds_in_hycom_mask = xr.where(~np.isnan(ds_in_hycom['ice_u_rot'][0,:,:].values), 1, 0) 
# ds_in_hycom['mask'] = (('lat', 'lon'), ds_in_hycom_mask)

# # Output grid (ROMS u)
# ds_out_u['mask'] = (('eta_u', 'xi_u'), ds_out_u.mask_u.values)

# # Output grid (ROMS v grid)
# ds_out_v['mask'] = (('eta_v', 'xi_v'), ds_out_v.mask_v.values)

# # Output grid (ROMS rho grid)
# ds_out_rho['mask'] = (('eta_rho', 'xi_rho'), ds_out_rho.mask_rho.values)

# # Regrid from HYCOM grid to u grid with the masks included and extrapolation used 
# regridder_hycom2u = xe.Regridder(ds_in_hycom, ds_out_u, method="bilinear", extrap_method='nearest_s2d') #extrap_method="nearest_s2d"
# regridder_hycom2u

# # Regrid from HYCOM grid to v grid with the masks included and extrapolation used 
# regridder_hycom2v = xe.Regridder(ds_in_hycom, ds_out_v, method="bilinear", extrap_method='nearest_s2d') #extrap_method="nearest_s2d"
# regridder_hycom2v

# # Regrid from HYCOM grid to rho grid with the masks included and extrapolation used 
# regridder_hycom2rho = xe.Regridder(ds_in_hycom, ds_out_rho, method="bilinear", extrap_method='nearest_s2d') #extrap_method="nearest_s2d"
# regridder_hycom2rho

In [52]:
# # Now use the regridder/weights to regrid the pre-rotated water u 
# ice_u_rot_on_roms_u = hycom_ice_data['ice_u_rot'].copy()
# ice_u_rot_on_roms_u = regridder_hycom2u(ice_u_rot_on_roms_u[4307,:,:])
# ice_u_rot_on_roms_u

In [53]:
# # Now use the regridder/weights to regrid the pre-rotated water v
# ice_v_rot_on_roms_v = hycom_ice_data['ice_v_rot'].copy()
# ice_v_rot_on_roms_v = regridder_hycom2v(ice_v_rot_on_roms_v[4307,:,:])
# ice_v_rot_on_roms_v

In [54]:
# # Interpolate ice area fraction onto rho points
# aice_on_rho = hycom_ice_data['sic'].copy()
# aice_on_rho = regridder_hycom2rho(aice_on_rho[4307,:,:])
# aice_on_rho

In [55]:
# # Interpolate ice thickness onto rho points 
# ice_thick_on_rho = hycom_ice_data['sih'].copy()
# ice_thick_on_rho = regridder_hycom2rho(ice_thick_on_rho[4307,:,:])
# ice_thick_on_rho

In [56]:
# # Plot these briefly to see what they look like 
# import matplotlib.pyplot as plt
# import cartopy
# import cartopy.crs as ccrs
# from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
# import cartopy.feature as cfeature
# import cmocean.cm as cmo
# import matplotlib.ticker as tick
# import warnings
# from matplotlib import ticker
# crs = ccrs.PlateCarree()
# warnings.filterwarnings("ignore") #turns off annoying warnings
# #Cartopy
# land_10m = cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                 edgecolor='face',
#                                 facecolor=cfeature.COLORS['land'])

# fig, ax = plt.subplots(4, 1, figsize=(12,4.5), dpi=200,
#                        constrained_layout=True, 
#                        subplot_kw={'projection': crs})

# # Numerical mixing of salinity 
# m1 = ice_u_rot_on_roms_u[:,:].plot(
#     x='lon_u', y='lat_u',
#     ax=ax[0],
#     cmap=cmo.speed, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#    vmin = -0.1,vmax=0.1
# )
# cbar_ax1 = fig.add_axes([0.7,0.80,0.015,0.2])
# fig.colorbar(m1,ax=ax[0],extend='both',
#              label=r'U',
#              pad=0.03, cax=cbar_ax1)

# # Set extent and map features
# ax[0].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[0].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[0].coastlines(resolution='10m', linewidth=.7)

# gl = ax[0].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# # Add a point in the white space to try to see if it is nan or not 
# ax[0].scatter(grid.lon_rho[118,350].values, grid.lat_rho[118,350].values, marker='x', s=5, color='deeppink')

# ax[0].set_title('Sea Ice U-Velocity')


# # Numerical mixing of temperature 
# m2 = ice_v_rot_on_roms_v[:,:].plot(
#     x='lon_v', y='lat_v',
#     ax=ax[1],
#     cmap=cmo.speed, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = -0.1,vmax=0.1
# )
# cbar_ax2 = fig.add_axes([0.7,0.55,0.015,0.2])
# fig.colorbar(m2,ax=ax[1],extend='both',
#              label=r'V',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[1].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[1].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[1].coastlines(resolution='10m', linewidth=.7)

# gl = ax[1].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[1].set_title('Sea Ice V-Velocity')


# # Sea ice area fraction
# m2 = aice_on_rho[:,:].plot(
#     x='lon_rho', y='lat_rho',
#     ax=ax[2],
#     cmap=cmo.ice, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = 0,vmax=1,
# )
# cbar_ax2 = fig.add_axes([0.7,0.25,0.015,0.2])
# fig.colorbar(m2,ax=ax[2],extend='both',
#              label=r'Ice Area Fraction',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[2].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[2].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[2].coastlines(resolution='10m', linewidth=.7)

# gl = ax[2].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[2].set_title('Sea Ice Area Fraction')


# # Sea ice thickness
# m2 = ice_thick_on_rho[:,:].plot(
#     x='lon_rho', y='lat_rho',
#     ax=ax[3],
#     cmap=cmo.dense, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = 0,vmax=3,
# )
# cbar_ax2 = fig.add_axes([0.7,0.03,0.015,0.2])
# fig.colorbar(m2,ax=ax[3],extend='both',
#              label=r'Ice Thickness',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[3].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[3].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[3].coastlines(resolution='10m', linewidth=.7)

# gl = ax[3].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[3].set_title('Sea Ice Thickness')



# fig.canvas.draw()



In [57]:
# # Check for nans...(and do min/max/mean/nans/general quality check things)
# # U
# print('ice_u_rot_on_roms_u nan: ', np.where(np.isnan(ice_u_rot_on_roms_u)))
# print('ice_u_rot_on_roms_u min: ', np.min(ice_u_rot_on_roms_u).values)
# print('ice_u_rot_on_roms_u max: ', np.max(ice_u_rot_on_roms_u).values)
# print('ice_u_rot_on_roms_u mean: ', np.mean(ice_u_rot_on_roms_u).values)

# # V
# print('ice_v_rot_on_roms_v nan: ', np.where(np.isnan(ice_v_rot_on_roms_v)))
# print('ice_v_rot_on_roms_v min: ', np.min(ice_v_rot_on_roms_v).values)
# print('ice_v_rot_on_roms_v max: ', np.max(ice_v_rot_on_roms_v).values)
# print('ice_v_rot_on_roms_v mean: ', np.mean(ice_v_rot_on_roms_v).values)

# # Ice area 
# print('aice_on_rho nan: ', np.where(np.isnan(aice_on_rho)))
# print('aice_on_rho min: ', np.min(aice_on_rho).values)
# print('aice_on_rho max: ', np.max(aice_on_rho).values)
# print('aice_on_rho mean: ', np.mean(aice_on_rho).values)

# # Ice thickness
# print('ice_thick_on_rho nan: ', np.where(np.isnan(ice_thick_on_rho)))
# print('ice_thick_on_rho min: ', np.min(ice_thick_on_rho).values)
# print('ice_thick_on_rho max: ', np.max(ice_thick_on_rho).values)
# print('ice_thick_on_rho mean: ', np.mean(ice_thick_on_rho).values)


In [58]:
# # Check to see if white patch is nan
# ice_u_rot_on_roms_u[118,350].values

# # It is...

In [59]:
# ice_u_rot_on_roms_u

In [60]:
# # Try extrapolating to get rid of nans?
# ice_u_rot_on_roms_u_nonan = ice_u_rot_on_roms_u.interpolate_na(dim='xi_u', method='linear', fill_value='extrapolate')
# ice_v_rot_on_roms_v_nonan = ice_v_rot_on_roms_v.interpolate_na(dim='xi_v', method='linear', fill_value='extrapolate')
# aice_on_rho_nonan = aice_on_rho.interpolate_na(dim='xi_rho', method='linear', fill_value='extrapolate')
# ice_thick_on_rho_nonan = ice_thick_on_rho.interpolate_na(dim='xi_rho', method='linear', fill_value='extrapolate')

In [61]:
# fig, ax = plt.subplots(4, 1, figsize=(12,4.5), dpi=200,
#                        constrained_layout=True, 
#                        subplot_kw={'projection': crs})

# # Numerical mixing of salinity 
# m1 = ice_u_rot_on_roms_u_nonan[:,:].plot(
#     x='lon_u', y='lat_u',
#     ax=ax[0],
#     cmap=cmo.speed, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#    vmin = -0.1,vmax=0.1
# )
# cbar_ax1 = fig.add_axes([0.7,0.80,0.015,0.2])
# fig.colorbar(m1,ax=ax[0],extend='both',
#              label=r'U',
#              pad=0.03, cax=cbar_ax1)

# # Set extent and map features
# ax[0].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[0].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[0].coastlines(resolution='10m', linewidth=.7)

# gl = ax[0].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# # Add a point in the white space to try to see if it is nan or not 
# ax[0].scatter(grid.lon_rho[118,350].values, grid.lat_rho[118,350].values, marker='x', s=5, color='deeppink')

# ax[0].set_title('Sea Ice U-Velocity')


# # Numerical mixing of temperature 
# m2 = ice_v_rot_on_roms_v_nonan[:,:].plot(
#     x='lon_v', y='lat_v',
#     ax=ax[1],
#     cmap=cmo.speed, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = -0.1,vmax=0.1
# )
# cbar_ax2 = fig.add_axes([0.7,0.55,0.015,0.2])
# fig.colorbar(m2,ax=ax[1],extend='both',
#              label=r'V',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[1].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[1].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[1].coastlines(resolution='10m', linewidth=.7)

# gl = ax[1].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[1].set_title('Sea Ice V-Velocity')


# # Sea ice area fraction
# m2 = aice_on_rho_nonan[:,:].plot(
#     x='lon_rho', y='lat_rho',
#     ax=ax[2],
#     cmap=cmo.ice, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = 0,vmax=1,
# )
# cbar_ax2 = fig.add_axes([0.7,0.25,0.015,0.2])
# fig.colorbar(m2,ax=ax[2],extend='both',
#              label=r'Ice Area Fraction',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[2].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[2].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[2].coastlines(resolution='10m', linewidth=.7)

# gl = ax[2].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[2].set_title('Sea Ice Area Fraction')


# # Sea ice thickness
# m2 = ice_thick_on_rho_nonan[:,:].plot(
#     x='lon_rho', y='lat_rho',
#     ax=ax[3],
#     cmap=cmo.dense, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = 0,vmax=3,
# )
# cbar_ax2 = fig.add_axes([0.7,0.03,0.015,0.2])
# fig.colorbar(m2,ax=ax[3],extend='both',
#              label=r'Ice Thickness',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[3].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[3].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[3].coastlines(resolution='10m', linewidth=.7)

# gl = ax[3].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[3].set_title('Sea Ice Thickness')



# fig.canvas.draw()

That looks super crude but is maybe better than nothing? Maybe try 2d bilinear interpolation?

In [62]:
# # Check the new values
# # U
# print('ice_u_rot_on_roms_u nan: ', np.where(np.isnan(ice_u_rot_on_roms_u_nonan)))

# # V
# print('ice_v_rot_on_roms_v nan: ', np.where(np.isnan(ice_v_rot_on_roms_v_nonan)))

# # Ice area 
# print('aice_on_rho nan: ', np.where(np.isnan(aice_on_rho_nonan)))

# # Ice thickness
# print('ice_thick_on_rho nan: ', np.where(np.isnan(ice_thick_on_rho_nonan)))

# # These still have nans so we could interpolate again over the other dimension and see if this fixes that?

In [63]:
# # Manually fill the remaining values just to get something to start with
# ice_u_rot_on_roms_u_nonan = ice_u_rot_on_roms_u_nonan.fillna(0.0)
# ice_v_rot_on_roms_v_nonan = ice_v_rot_on_roms_v_nonan.fillna(0.0)
# aice_on_rho_nonan = aice_on_rho_nonan.fillna(0.0)
# ice_thick_on_rho_nonan = ice_thick_on_rho_nonan.fillna(0.0)

In [64]:
# # Check the new values
# # U
# print('ice_u_rot_on_roms_u nan: ', np.where(np.isnan(ice_u_rot_on_roms_u_nonan)))

# # V
# print('ice_v_rot_on_roms_v nan: ', np.where(np.isnan(ice_v_rot_on_roms_v_nonan)))

# # Ice area 
# print('aice_on_rho nan: ', np.where(np.isnan(aice_on_rho_nonan)))

# # Ice thickness
# print('ice_thick_on_rho nan: ', np.where(np.isnan(ice_thick_on_rho_nonan)))

In [65]:
# # Check again to see what this filled version looks like 

# fig, ax = plt.subplots(4, 1, figsize=(12,4.5), dpi=200,
#                        constrained_layout=True, 
#                        subplot_kw={'projection': crs})

# # Numerical mixing of salinity 
# m1 = ice_u_rot_on_roms_u_nonan[:,:].plot(
#     x='lon_u', y='lat_u',
#     ax=ax[0],
#     cmap=cmo.speed, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#    vmin = -0.1,vmax=0.1
# )
# cbar_ax1 = fig.add_axes([0.7,0.80,0.015,0.2])
# fig.colorbar(m1,ax=ax[0],extend='both',
#              label=r'U',
#              pad=0.03, cax=cbar_ax1)

# # Set extent and map features
# ax[0].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[0].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[0].coastlines(resolution='10m', linewidth=.7)

# gl = ax[0].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# # Add a point in the white space to try to see if it is nan or not 
# #ax[0].scatter(grid.lon_rho[118,350].values, grid.lat_rho[118,350].values, marker='x', s=5, color='deeppink')

# ax[0].set_title('Sea Ice U-Velocity')


# # Numerical mixing of temperature 
# m2 = ice_v_rot_on_roms_v_nonan[:,:].plot(
#     x='lon_v', y='lat_v',
#     ax=ax[1],
#     cmap=cmo.speed, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = -0.1,vmax=0.1
# )
# cbar_ax2 = fig.add_axes([0.7,0.55,0.015,0.2])
# fig.colorbar(m2,ax=ax[1],extend='both',
#              label=r'V',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[1].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[1].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[1].coastlines(resolution='10m', linewidth=.7)

# gl = ax[1].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[1].set_title('Sea Ice V-Velocity')


# # Sea ice area fraction
# m2 = aice_on_rho_nonan[:,:].plot(
#     x='lon_rho', y='lat_rho',
#     ax=ax[2],
#     cmap=cmo.ice, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = 0,vmax=1,
# )
# cbar_ax2 = fig.add_axes([0.7,0.25,0.015,0.2])
# fig.colorbar(m2,ax=ax[2],extend='both',
#              label=r'Ice Area Fraction',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[2].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[2].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[2].coastlines(resolution='10m', linewidth=.7)

# gl = ax[2].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[2].set_title('Sea Ice Area Fraction')


# # Sea ice thickness
# m2 = ice_thick_on_rho_nonan[:,:].plot(
#     x='lon_rho', y='lat_rho',
#     ax=ax[3],
#     cmap=cmo.dense, transform=ccrs.PlateCarree(),
#     add_colorbar=False, facecolor="gray",
#     vmin = 0,vmax=3,
# )
# cbar_ax2 = fig.add_axes([0.7,0.03,0.015,0.2])
# fig.colorbar(m2,ax=ax[3],extend='both',
#              label=r'Ice Thickness',
#              pad=0.03, cax=cbar_ax2)

# # Set extent and map features
# ax[3].set_extent([-153.5,-141.4,69.75, 71.5],ccrs.PlateCarree())
# ax[3].add_feature(cfeature.NaturalEarthFeature('physical', 'land', '10m',
#                                             facecolor='0.8'), linewidth=.7)
# ax[3].coastlines(resolution='10m', linewidth=.7)

# gl = ax[3].gridlines(crs=ccrs.PlateCarree(), 
#                   draw_labels=True, 
#                   x_inline=False, y_inline=False, 
#                   linewidth=0.33, color='k',alpha=0.5)
# gl.right_labels = gl.top_labels = False
# gl.xlocator = ticker.FixedLocator([-152, -150,-148,-146,-144,-142])
# gl.xlabel_style = {'rotation': 0, 'ha': 'center'}

# ax[3].set_title('Sea Ice Thickness')



# fig.canvas.draw()

Other Options for filling nans:
- Fancy way from bryclm scripts that uses Fortran 
- Interpolate again over other direction
- Try another interpolation method 


In [66]:
# Try fancy way done in bry clm files using Fortran 


In [67]:
# # Use fill.f90 to fill the nans in the array
# # Import fill.f90 from model2roms to see how to 
# # use this/if it can be used to get rid of nans 
# from numpy import f2py
# with open('/global/homes/b/bundzis/Projects/Beaufort_ROMS_2020_dvd_myroms_ice/Scripts/Forcing_files/Bryclm/fill.f90') as sourcefile2:
#     sourcecode2 = sourcefile2.read()
# f2py.compile(sourcecode2, modulename='fill', extension='.f90')
# import fill

In [68]:
# # Define a function to call to do the filling, taken from model2roms
# def laplacefilter(field, threshold, toxi, toeta):
#     undef = 2.0e+35 
#     tx = 0.9 * undef
#     critx = 0.01
#     cor = 1.6
#     mxs = 10

#     field = np.where(abs(field) > threshold, undef, field)

#     field = fill.extrapolate.fill(int(1), int(toxi),
#                                 int(1), int(toeta),
#                                 float(tx), float(critx), float(cor), float(mxs),
#                                 np.asarray(field, order='F'),
#                                 int(toxi),
#                                 int(toeta))
#     return field

In [69]:
# # Interpolate again over other direction to try to get rid of nans 
# ice_u_rot_on_roms_u_nonan = ice_u_rot_on_roms_u_nonan.interpolate_na(dim='eta_u', method='linear')
# ice_v_rot_on_roms_v_nonan = ice_v_rot_on_roms_v_nonan.interpolate_na(dim='eta_v', method='linear')
# aice_on_rho_nonan = aice_on_rho_nonan.interpolate_na(dim='eta_rho', method='linear')
# ice_thick_on_rho_nonan = ice_thick_on_rho_nonan.interpolate_na(dim='eta_rho', method='linear')


In [70]:
# # Check the new values
# # U
# print('ice_u_rot_on_roms_u nan: ', np.where(np.isnan(ice_u_rot_on_roms_u_nonan)))

# # V
# print('ice_v_rot_on_roms_v nan: ', np.where(np.isnan(ice_v_rot_on_roms_v_nonan)))

# # Ice area 
# print('aice_on_rho nan: ', np.where(np.isnan(aice_on_rho_nonan)))

# # Ice thickness
# print('ice_thick_on_rho nan: ', np.where(np.isnan(ice_thick_on_rho_nonan)))

In [71]:
# # Try scipy 
# import scipy 
# ice_u_rot_on_roms_u_nonan2d = scipy.interpolate.RegularGridInterpolator()  interp2d(ice_u_rot_on_roms_u.eta_u, ice_u_rot_on_roms_u.xi_u, ice_u_rot_on_roms_u, kind='linear')
# ice_v_rot_on_roms_v_nonan2d = ice_v_rot_on_roms_v.interpolate_na(dim='xi_v', method='linear')
# aice_on_rho_nonan2d = aice_on_rho.interpolate_na(dim='xi_rho', method='linear')
# ice_thick_on_rho_nonan2d = ice_thick_on_rho.interpolate_na(dim='xi_rho', method='linear')

In [72]:
# Make some variables with 0s for variables that ROMS
# wants but we don't have data for
# Meltpond thickness (meltpond_thickness)
meltpond_thickness_tmp = np.zeros_like((Aice_tmp))

# Ice age (ice_age)
ice_age_tmp = np.zeros_like((Aice_tmp))

# Snow thickness (snow_thickness)
snow_thick_tmp = np.zeros_like((Aice_tmp))

# Sea ice temperature (Tice)
ice_temp_tmp = np.zeros_like((Aice_tmp))

# Under ice temperature (under_ice_temp)
under_ice_temp_tmp = np.zeros_like((Aice_tmp))

# Under ice salinity (under_ice_salt)
under_ice_salt_tmp = np.zeros_like((Aice_tmp))

# Sea ice surface temperature  (ice_sst)
sea_ice_surface_temperature_tmp = np.zeros_like((Aice_tmp))

# Sea ice internal stress - xx (ice_Sxx)
ice_Sxx_tmp = np.zeros_like((Aice_tmp))

# Sea ice internal stress - xy (ice_Sxy)
ice_Sxy_tmp = np.zeros_like((Aice_tmp))

# Sea ice internal stress - yy (ice_Syy)
ice_Syy_tmp = np.zeros_like((Aice_tmp))



In [73]:
np.shape(ice_temp_tmp)

(1, 206, 608)

In [74]:
# # Sea Ice
# # Set the values to be used in the initial conditions file
# # Needed if not reading in from climatology but using this section instead

# # Ice area fraction (Aice)
# # fraction of cell covered by ice
# Aice_tmp = aice_on_rho_nonan

# # Ice thickness (ice_thickness)
# # average ice thickness in cell
# ice_thickness_tmp = ice_thick_on_rho_nonan

# # Ice u-velocity (Uice)
# # u-component of ice velocity 
# Uice_tmp = ice_u_rot_on_roms_u_nonan

# # Ice v-velocity (Vice)
# # v-component of ice velocity
# Vice_tmp = ice_v_rot_on_roms_v_nonan

In [75]:
# print(salt_tmp.shape)
# print(zeta_tmp.shape)
print(time_tmp_len)
print(s_rho_tmp)
print(Mp)
print(Lp)
# print(Cs_r_tmp)
#print(hc_tmp)
#print(Cs_r_tmp)

1
30
206
608


In [76]:
#print(hc_tmp)
#print(Cs_r_tmp)
#nc1.close()
#nc2.close()

In [77]:
# Before making the netcdf, CHECK FOR NANS
# Are there nans in anything? Especially things from dbSeabed data?
# If so, replace nans with -999
# Maybe ignore this for now because this was fixed in the dbSeabed data
# But definitely check all of the input files before running ROMS
# using nan_check.py

### Passive Tracers
Set all passive tracers to zero!

In [78]:
# # Passive Tracers - real values
# # Set the initial conditions for the passive tracers
# # used for numerical mixing 

# # Dye_01 (salinity)
# dye_01_tmp = salt_tmp.copy()

# # Dye_02 (salt^2)
# dye_02_tmp = dye_01_tmp**2

# # Dye_03 (numerical mixing)
# dye_03_tmp = np.zeros_like((dye_02_tmp))

In [79]:
# Passive Tracers - zeros
# Set the initial conditions for the passive tracers
# used for numerical mixing 

# Dye_01 (salinity)
dye_01_tmp = np.zeros_like((salt_tmp))

# Dye_02 (salt^2)
dye_02_tmp = np.zeros_like((salt_tmp))

# Dye_03 (numerical mixing salinity)
dye_03_tmp = np.zeros_like((salt_tmp))

# Dye_04 (temperature)
dye_04_tmp = np.zeros_like((salt_tmp))

# Dye_05 (temperature^2)
dye_05_tmp = np.zeros_like((salt_tmp))

# Dye_06 (numerical mixing temperature)
dye_06_tmp = np.zeros_like((salt_tmp))


In [80]:
# Check if the tracer things worked 
print('dye_01[0,10,100,100]: ', dye_01_tmp[0,10,100,100])
print('dye_02[0,10,100,100]: ', dye_02_tmp[0,10,100,100])
print('dye_03[0,10,100,100]: ', dye_03_tmp[0,10,100,100])
print('dye_04[0,10,100,100]: ', dye_04_tmp[0,10,100,100])
print('dye_05[0,10,100,100]: ', dye_05_tmp[0,10,100,100])
print('dye_06[0,10,100,100]: ', dye_06_tmp[0,10,100,100])
print(np.shape(dye_01_tmp))

dye_01[0,10,100,100]:  0.0
dye_02[0,10,100,100]:  0.0
dye_03[0,10,100,100]:  0.0
dye_04[0,10,100,100]:  0.0
dye_05[0,10,100,100]:  0.0
dye_06[0,10,100,100]:  0.0
(1, 30, 206, 608)


### Create netcdf file and save information about it

In [81]:
# Name of file I am writing to
#init_cond = '/global/homes/b/bundzis/Projects/Beaufort_ROMS_2020_dvd_myroms_ice/Include/initial_conds_beaufort_shelf_sea_ice_jul_2020_002.nc' 
#init_cond = '/global/homes/b/bundzis/Projects/Beaufort_ROMS_2020_dvd_myroms_ice/Include/initial_conds_beaufort_shelf_sea_ice_sep_2019_20vert_002.nc' 
init_cond = '/global/homes/b/bundzis/Projects/Beaufort_ROMS_2020_dvd_myroms_ice_30vert/Include/initial_conds_beaufort_shelf_sea_ice_sep_2019_30vert_002.nc' 


# Create file to write to
nc = Dataset(init_cond, 'w', format='NETCDF4')

# Global attributes
global_defaults = dict(gridname = 'KakAKgrd_shelf_big010_smooth006_thin_sponge.nc',
                      type = 'ROMS initial conditions forcing file',
                      history = 'Created by Brianna Undzis',
                      Conventions = 'CF',
                      Institution = 'University of Colorado Boulder',
                      date = str(datetime.today()))

# Create dictionary for model
d = {}
d = global_defaults

for att, value in d.items():
    setattr(nc, att, value)

# Create dimensions
nc.createDimension('xi_psi', L)
nc.createDimension('xi_rho', Lp)
nc.createDimension('xi_u', L)
nc.createDimension('xi_v', Lp)
nc.createDimension('eta_psi', M)
nc.createDimension('eta_rho', Mp)
nc.createDimension('eta_u', Mp)
nc.createDimension('eta_v', M)
nc.createDimension('s_rho', s_rho_tmp)
nc.createDimension('s_w', s_w_tmp_len)
nc.createDimension('tracer', tracer_tmp_len)
nc.createDimension('Nbed', Nbed_tmp)
nc.createDimension('time', None)
nc.createDimension('one', 1)
nc.createDimension('two', 2)

<class 'netCDF4._netCDF4.Dimension'>: name = 'two', size = 2

### Fill dimensions and variables

In [82]:
#Fill dimensions
# xi_rho = nc.createVariable('xi_rho', 'd', ('xi_rho',), zlib=True)
# xi_rho.long_name = "X coordinate of RHO-points"
# xi_rho.standard_name = "projection_x_coordinate"
# xi_rho.units = "meter"
# xi_rho[:] = xi_rho_tmp[:]

# eta_rho = nc.createVariable('eta_rho', 'd', ('eta_rho',), zlib=True)
# eta_rho.long_name = "Y coordinate of RHO-points"
# eta_rho.standard_name = "projection_y_coordinate"
# eta_rho.units = "meter"
# eta_rho[:] = eta_rho_tmp[:]

In [83]:
# Fill variables
# theta_b
theta_b = nc.createVariable('theta_b', 'f8', ('one',), zlib=True)
theta_b.long_name = 'S-coordinate bottom control parameter'
theta_b.units = '1'
theta_b[:] = theta_b_tmp

# theta_s 
theta_s = nc.createVariable('theta_s', 'f8', ('one',), zlib=True)
theta_s.long_name = 'S-coordinate surface control parameter'
theta_s.units = '1'
theta_s[:] = theta_s_tmp

# Tcline
Tcline = nc.createVariable('Tcline', 'f8', ('one',), zlib=True)
Tcline.long_name = 'S-coordinate surface/bottom layer width'
Tcline.units = 'meter'
Tcline[:] = Tcline_tmp

# hc
hc = nc.createVariable('hc', 'f8', ('one',), zlib=True)
hc.long_name = 'S-coordinate parameter, critical depth'
hc.units = '1'
hc[:] = hc_tmp

# # Cs_r - commented out since idk what this is for 30-layers 
# Cs_r = nc.createVariable('Cs_r', 'f8', ('s_rho',), zlib=True)
# Cs_r.long_name = 'S-coordinate stretching curves at RHO-points'
# Cs_r.units = '1'
# Cs_r.valid_min = -1.0
# Cs_r.valid_max = 0.0
# Cs_r.field = 'Cs_r, scalar'
# Cs_r[:] = Cs_r_tmp

# # Cs_w - commented out since idk what this is for 30-layers 
# Cs_w = nc.createVariable('Cs_w', 'f8', ('s_w',), zlib=True)
# Cs_w.long_name = 'S-coordinate stretching curves at W-points'
# Cs_w.units = '1'
# Cs_w.valid_min = -1.0
# Cs_w.valid_max = 0.0
# Cs_w.field = 'Cs_w, scalar'
# Cs_w[:] = Cs_w_tmp

# # sc_r
# sc_r = nc.createVariable('sc_r', 'f8', ('s_rho',), zlib=True)
# sc_r.long_name = 'S-coordinate at RHO-points'
# sc_r.units = '1'
# sc_r.valid_min = -1.0
# sc_r.valid_max = 0.0
# sc_r.field = 's_rho, scalar'
# sc_r[:] = sc_r_tmp

# # sc_w
# sc_w = nc.createVariable('sc_w', 'f8', ('s_w',), zlib=True)
# sc_w.long_name = 'S-coordinate at W-points'
# sc_w.units = '1'
# sc_w.valid_min = -1.0
# sc_w.valid_max = 0.0
# sc_w.field = 's_w, scalar'
# sc_w[:] = sc_w_tmp

# ocean_time
ocean_time = nc.createVariable('ocean_time', 'f8', ('time',), zlib=True)
ocean_time.long_name = 'seconds since 2000-01-01 00:00:00'
ocean_time.units = 'second'
ocean_time.field = 'ocean_time, scalar, series'
#ocean_time[:] = ocean_time_tmp[:]
ocean_time[:] = ocean_time_tmp

# salinity
salt = nc.createVariable('salt', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
salt.long_name = 'salinity functional'
salt.units = 'PSS'
salt.field = 'salinity, scalar, series'
salt[:,:,:,:] = salt_tmp[:,:,:,:]
print(salt[0,0,0,0]) 
print(salt_tmp[0,0,0,0])

# temperature
temp = nc.createVariable('temp', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
temp.long_name = 'potential temperature functional'
temp.units = 'C'
temp.field = 'temperature, scalar, series'
temp[:,:,:,:] = temp_tmp[:,:,:,:]

# u
u = nc.createVariable('u', 'f8', ('time', 's_rho', 'eta_u', 'xi_u'), zlib=True)
u.long_name = 'u-momentum component'
u.units = 'meter second-1'
u.field = 'u-velocty, scalar, series'
u[:,:,:,:] = u_tmp[:,:,:,:]

# ubar
ubar = nc.createVariable('ubar', 'f8', ('time', 'eta_u', 'xi_u'), zlib=True)
ubar.long_name = 'vertically integrated u-momentum component'
ubar.units = 'meter second-1'
ubar.field = 'ubar-velocity, scalar, series'
ubar[:,:,:] = ubar_tmp[:,:,:]

# v
v = nc.createVariable('v', 'f8', ('time', 's_rho', 'eta_v', 'xi_v'), zlib=True)
v.long_name = 'v-momentum component'
v.units = 'meter second-1'
v.field = 'v-velocity, scalar, series'
v[:,:,:,:] = v_tmp[:,:,:,:]

# vbar
vbar = nc.createVariable('vbar', 'f8', ('time', 'eta_v', 'xi_v'), zlib=True)
vbar.long_name = 'vertically integrated v-momentum component'
vbar.units = 'meter second-1'
vbar.field = 'vbar-velocity, scalar, series'
vbar[:,:,:] = vbar_tmp[:,:,:]

# zeta
zeta = nc.createVariable('zeta', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
zeta.long_name = 'free-surface'
zeta.units = 'meter'
zeta.field = 'free-surface, scalar, series'
zeta[:,:,:] = zeta_tmp[:,:,:]

# bathymetry
bath = nc.createVariable('bath', 'f8', ('eta_rho', 'xi_rho'), zlib=True)
bath.long_name = 'bathymetry'
bath.units = 'meter'
bath.field = 'bathymetry, scalar, series'
bath[:,:] = bath_tmp[:,:]

# Sea Ice
# Ice area fraction (Aice)
Aice_g = nc.createVariable('Aice', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
Aice_g.standard_name = 'sea_ice_area_fraction'
Aice_g.long_name = 'fraction of cell covered by ice'
Aice_g.units = 'nondimensional'
Aice_g[:,:,:] = Aice_tmp 

# Ice thickness (ice_thickness)
ice_thickness_g = nc.createVariable('ice_thickness', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
ice_thickness_g.standard_name = 'sea_ice_thickness'
ice_thickness_g.long_name = 'average ice thickness in cell'
ice_thickness_g.units = 'meter'
ice_thickness_g[:,:,:] = ice_thickness_tmp 

# # Meltpond thickness (meltpond_thickness)
# meltpond_thickness_g = nc.createVariable('meltpond_thickness', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# meltpond_thickness_g.standard_name = 'melt_pond_water_thickness_on_sea_ice'
# meltpond_thickness_g.long_name = 'surface melt water thickness on ice'
# meltpond_thickness_g.units = 'meter'
# meltpond_thickness_g[:,:,:] = meltpond_thickness_tmp 

# # Ice age (ice_age)
# ice_age_g = nc.createVariable('ice_age', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# ice_age_g.standard_name = 'age_of_sea_ice'
# ice_age_g.long_name = 'sage of sesa ice'
# ice_age_g.units = 'second'
# ice_age_g[:,:,:] = ice_age_tmp 

# # Snow thickness (snow_thickness)
# snow_thickness_g = nc.createVariable('snow_thickness', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# snow_thickness_g.standard_name = 'snowfall_thickness_above_sea_ice'
# snow_thickness_g.long_name = 'thickness of snow cover'
# snow_thickness_g.units = 'meter'
# snow_thickness_g[:,:] = snow_thick_tmp 

# # Sea ice temperature (sea_ice_temperature)
# tice_g = nc.createVariable('Tice', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# tice_g.standard_name = 'sea_ice_temperature'
# tice_g.long_name = 'interior ice temperature'
# tice_g.units = 'Celcius'
# tice_g[:,:] = ice_temp_tmp

# # Under ice temperature (under_ice_temp)
# under_ice_temp_g = nc.createVariable('under_ice_temp', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# under_ice_temp_g.standard_name = 'temperature_of_molecular_sub_layer_under_sea_ice'
# under_ice_temp_g.long_name = 'temperature of molecular sub-layer under ice'
# under_ice_temp_g.units = 'Celcius'
# under_ice_temp_g[:,:] = under_ice_temp_tmp

# # Under ice salinity (under_ice_salt)
# under_ice_salt_g = nc.createVariable('under_ice_salt', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# under_ice_salt_g.standard_name = 'salinity_of_molecular_sub_layer_under_sea_ice'
# under_ice_salt_g.long_name = 'salinity of molecular sub-layer under ice'
# under_ice_salt_g.units = 'Celcius'
# under_ice_salt_g[:,:] = under_ice_salt_tmp

# # Sea ice surface temperature  (ice_sst)
# sea_ice_surface_temperature_g = nc.createVariable('ice_sst', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# sea_ice_surface_temperature_g.standard_name = 'sea_ice_surface_temperature'
# sea_ice_surface_temperature_g.long_name = 'temperature of ice/snow surface'
# sea_ice_surface_temperature_g.units = 'Celcius'
# sea_ice_surface_temperature_g[:,:] = sea_ice_surface_temperature_tmp

# Ice u-velocity (Uice)
Uice_g = nc.createVariable('Uice', 'f8', ('time', 'eta_u', 'xi_u'), zlib=True)
Uice_g.standard_name = 'sea_ice_x_velocity'
Uice_g.long_name = 'u-component of ice velocity'
Uice_g.units = 'meter second-1'
Uice_g[:,:,:] = Uice_tmp 

# Ice v-velocity (Vice)
Vice_g = nc.createVariable('Vice', 'f8', ('time', 'eta_v', 'xi_v'), zlib=True)
Vice_g.standard_name = 'sea_ice_y_velocity'
Vice_g.long_name = 'v-component of ice velocity'
Vice_g.units = 'meter second-1'
Vice_g[:,:,:] = Vice_tmp 

# # Sea ice internal stress - x (ice_Sxx)
# ice_Sxx_g = nc.createVariable('ice_Sxx', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# ice_Sxx_g.standard_name = 'sea_ice_internal_xx_stress'
# ice_Sxx_g.long_name = 'internal ice stress xx-component'
# ice_Sxx_g.units = 'Newton meter-1'
# ice_Sxx_g[:,:,:] = ice_Sxx_tmp 

# # Sea ice internal stress - xy (ice_Sxy)
# ice_Sxy_g = nc.createVariable('ice_Sxy', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# ice_Sxy_g.standard_name = 'sea_ice_internal_xy_stress'
# ice_Sxy_g.long_name = 'internal ice stress xy-component'
# ice_Sxy_g.units = 'Newton meter-1'
# ice_Sxy_g[:,:,:] = ice_Sxy_tmp 

# # Sea ice internal stress - yy (ice_Sxy)
# ice_Syy_g = nc.createVariable('ice_Syy', 'f8', ('time', 'eta_rho', 'xi_rho'), zlib=True)
# ice_Syy_g.standard_name = 'sea_ice_internal_yy_stress'
# ice_Syy_g.long_name = 'internal ice stress yy-component'
# ice_Syy_g.units = 'Newton meter-1'
# ice_Syy_g[:,:,:] = ice_Syy_tmp 



# Passive tracers
# dye_01 (salinity)
dye_01_g = nc.createVariable('dye_01', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
dye_01_g.long_name = 'dye_01 concentration'
dye_01_g.units = 'kilogram meter-3'
dye_01_g[:,:,:,:] = dye_01_tmp 

# dye_02 (salinity^2)
dye_02_g = nc.createVariable('dye_02', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
dye_02_g.long_name = 'dye_02 concentration'
dye_02_g.units = 'kilogram meter-3'
dye_02_g[:,:,:,:] = dye_02_tmp 

# dye_03 (numerical mixing of salinity)
dye_03_g = nc.createVariable('dye_03', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
dye_03_g.long_name = 'dye_03 concentration'
dye_03_g.units = 'kilogram meter-3'
dye_03_g[:,:,:,:] = dye_03_tmp 

# dye_04 (temperature)
dye_04_g = nc.createVariable('dye_04', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
dye_04_g.long_name = 'dye_04 concentration'
dye_04_g.units = 'kilogram meter-3'
dye_04_g[:,:,:,:] = dye_04_tmp 

# dye_05 (temperature^2)
dye_05_g = nc.createVariable('dye_05', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
dye_05_g.long_name = 'dye_05 concentration'
dye_05_g.units = 'kilogram meter-3'
dye_05_g[:,:,:,:] = dye_05_tmp 

# dye_06 (numerical mixing of temperature)
dye_06_g = nc.createVariable('dye_06', 'f8', ('time', 's_rho', 'eta_rho', 'xi_rho'), zlib=True)
dye_06_g.long_name = 'dye_06 concentration'
dye_06_g.units = 'kilogram meter-3'
dye_06_g[:,:,:,:] = dye_06_tmp 


nc.close()

0.0
0.0


In [84]:
# Close the netcdf files
# grid file
nc1.close()

# restrat_mix_ini.nc file
#nc2.close()